# CourtSide — results and tests

A short report of **real recorded responses** from the same local Qwen model, with and without NBA retrieval. Outputs are saved so you can read the results directly on GitHub.

These cells read `docs/comparison.json`; they do not run model inference again. To record a fresh comparison, run `HF_HUB_OFFLINE=1 uv run python -m scripts.compare`. For the final test cell, use the project Python environment installed with `uv sync --python 3.12 --extra dev`.


In [1]:
# Load the recorded experiment; support opening from the repo or its parent.
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "docs/comparison.json").exists():
    ROOT = ROOT.parent
report = json.loads((ROOT / "docs/comparison.json").read_text())
results = report["results"]

print("Model:", report["model"])
print("Recorded:", report["recorded_at"])
print("Settings:", report["settings"])
print("Examples:", len(results))


def show_result(index):
    """Print exact saved answers and the evidence behind the RAG answer."""
    result = results[index]
    print("QUESTION:", result["question"])
    print("\nMODEL ONLY:\n" + result["model_only"]["answer"])
    mode = result["rag"]["answer_mode"]
    print("\nRAG APPLICATION (" + mode + "):\n" + result["rag"]["answer"])
    print("\nOFFICIAL EVIDENCE:")
    for source in result["rag"]["sources"]:
        print(f'[{source["citation"]}] {source["text"]}')
        print(source["url"])


Model: Qwen/Qwen2.5-0.5B-Instruct
Recorded: 2026-11-14T17:11:14.158317+00:00
Settings: {'do_sample': False, 'max_new_tokens': 240, 'repetition_penalty': 1.08, 'device': 'cpu'}
Examples: 3


## 1. Regular-season MVP

The model-only answer invents a winner. With the official season row, the model gives Shai Gilgeous-Alexander and a source citation.


In [2]:
# Compare the same MVP question, using the exact saved responses.
show_result(0)


QUESTION: Who won the regular-season MVP in 2024-25?

MODEL ONLY:
In 2024-25, the regular-season MVP was **Jared Goff**. He played for the New York Knicks and led them to a playoff appearance with a record of 18-16.

RAG APPLICATION (generated):
regular-season MVP — 2024-25 — Shai Gilgeous-Alexander, Oklahoma City Thunder [1]

OFFICIAL EVIDENCE:
[1] Regular-season MVP winners
2024-25 — Shai Gilgeous-Alexander, Oklahoma City Thunder ( Voting )
https://www.nba.com/news/history-mvp-award-winners


## 2. Rookie of the Year

Retrieval supplies the season-specific winner, Stephon Castle, instead of the unsupported model-only answer.


In [3]:
# The cited source is shown beneath the RAG response.
show_result(1)


QUESTION: Who won Rookie of the Year in 2024-25?

MODEL ONLY:
In 2024-25, the New Orleans Pelicans' rookie of the year was Klay Thompson. He was selected as the first overall pick by the team in the 2024 NBA draft.

RAG APPLICATION (generated):
Rookie of the Year winners - 2024-25 - Stephon Castle, San Antonio Spurs [1]

OFFICIAL EVIDENCE:
[1] Rookie of the Year winners
2024-25 — Stephon Castle, San Antonio Spurs ( Voting )
https://www.nba.com/news/history-rookie-of-the-year-winners


## 3. Champion and Finals MVP

This example uses the application's **verified-record fallback**. It shows the original matching NBA rows because the small model did not provide a complete cited answer. These rows are extracted evidence, not model-written prose.


In [4]:
# Keep the fallback label visible: this compares the full RAG application.
show_result(2)


QUESTION: Who won the 2023 NBA championship and Finals MVP?

MODEL ONLY:
The 2023 NBA championship was won by the Los Angeles Lakers, while the Finals MVP was LeBron James of the Cleveland Cavaliers.

RAG APPLICATION (verified_records):
Official archive records:
- NBA Finals MVP winners: 2022-23: Nikola Jokic, Denver Nuggets [1]
- NBA champions: 2022-23 — Denver Nuggets def. Miami Heat, 4-1 [2]

OFFICIAL EVIDENCE:
[1] NBA Finals MVP winners
2022-23: Nikola Jokic, Denver Nuggets
https://www.nba.com/news/history-finals-mvp-winners
[2] NBA champions
2022-23 — Denver Nuggets def. Miami Heat, 4-1
https://www.nba.com/news/history-nba-champions


## 4. Check the recorded results

Check the expected winners, official source URLs, and citation references. These are checks on three saved examples, not an accuracy benchmark. The RAG application also changes the prompt and applies record filtering and fallback logic.


In [5]:
import re
import unicodedata


def normalized(text):
    # Ignore accents and case when checking names such as Jokić / Jokic.
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().casefold()


expected = [
    ("Shai Gilgeous-Alexander",),
    ("Stephon Castle",),
    ("Denver Nuggets", "Nikola Jokic"),
]
assert len(results) == len(expected)
for result, winners in zip(results, expected):
    rag = result["rag"]
    assert all(normalized(name) in normalized(rag["answer"]) for name in winners)
    assert rag["sources"], "A factual result must have retrieved evidence."
    assert all(source["url"].startswith("https://www.nba.com/") for source in rag["sources"])
    source_text = " ".join(source["text"] for source in rag["sources"])
    assert all(normalized(name) in normalized(source_text) for name in winners)
    citations = {int(number) for number in re.findall(r"\[(\d+)\]", rag["answer"])}
    source_ids = {source["citation"] for source in rag["sources"]}
    assert citations and citations <= source_ids
    print("PASS:", result["question"])
print("\n3 saved-result checks passed: winners, evidence, and citation references.")


PASS: Who won the regular-season MVP in 2024-25?
PASS: Who won Rookie of the Year in 2024-25?
PASS: Who won the 2023 NBA championship and Finals MVP?

3 saved-result checks passed: winners, evidence, and citation references.


## 5. Run the project tests

The regression suite checks ingestion, API validation, season and player filtering, missing-season handling, and the verified-record fallback. It does not download or load model weights.


In [6]:
# Run the actual tests with this notebook's Python environment.
import subprocess
import sys

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--disable-warnings"],
    cwd=ROOT, text=True, capture_output=True,
)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr or completed.stdout


...............                                                          [100%]
15 passed, 1 warning in 7.57s



The saved examples show why source retrieval and evidence inspection matter for this small model. General model-generated answers can still be wrong. To try your own questions, start the web app with `./run.sh` and open http://localhost:8000.
